# Clasificación de sentimientos con TF-IDF y SGDClassifier

## 1. Introducción

En este notebook se implementa un clasificador de sentimientos usando `SGDClassifier` de scikit-learn.

La representación combina dos tipos de TF-IDF sobre la reseña completa:

1. **TF-IDF de palabras**, usando unigramas y bigramas.
2. **TF-IDF de caracteres**, usando `char_wb` con n-gramas de 3 a 5 caracteres.

`SGDClassifier` entrena modelos lineales mediante descenso de gradiente estocástico, por lo que resulta apropiado para matrices dispersas y de alta dimensionalidad como las producidas por TF-IDF.

Para que el resultado sea comparable con los demás modelos del equipo se utiliza la misma partición estratificada 80/20, `random_state=42` y validación cruzada estratificada de cinco folds. `eval.csv` se reserva únicamente para generar el archivo de Kaggle.


## 2. Importación de librerías


In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    GridSearchCV
)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    ConfusionMatrixDisplay
)

# Detectar la raíz del proyecto tanto si el notebook se ejecuta
# desde notebooks/ como desde la raíz.
raiz = Path.cwd().resolve()
if raiz.name == "notebooks":
    raiz = raiz.parent

print("Raíz del proyecto:", raiz)


## 3. Carga y partición de los datos

Se cargan las 12.000 reseñas de entrenamiento y las 3.000 reseñas de evaluación. El conjunto de validación contiene el 20% de `train.csv` y mantiene la proporción de clases mediante `stratify=y`.


In [ ]:
data_dir = raiz / "data"

train = pd.read_csv(data_dir / "train.csv")
eval_data = pd.read_csv(data_dir / "eval.csv")
sample = pd.read_csv(data_dir / "sample_submission.csv")

X = train["text"]
y = train["label"]

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Entrenamiento:", len(X_train))
print("Validación:", len(X_val))
print("Evaluación Kaggle:", len(eval_data))
print("\nDistribución entrenamiento:")
print(y_train.value_counts(normalize=True).round(3))


## 4. Representación TF-IDF de palabras y caracteres

Se combinan mediante `FeatureUnion`:

- **Palabras:** unigramas y bigramas. Los bigramas permiten capturar expresiones como `muy bueno`, `no sirve` o `lo recomiendo`.
- **Caracteres:** `char_wb` de 3 a 5 caracteres. Esta representación puede capturar raíces, sufijos, variaciones ortográficas e intensificadores sin depender exclusivamente de la tokenización por palabras.

En ambos casos se utiliza `min_df=2` para eliminar términos extremadamente raros y `sublinear_tf=True` para reducir el peso relativo de repeticiones muy frecuentes.


In [ ]:
representacion_sgd = FeatureUnion([
    (
        "palabras",
        TfidfVectorizer(
            analyzer="word",
            ngram_range=(1, 2),
            min_df=2,
            sublinear_tf=True
        )
    ),
    (
        "caracteres",
        TfidfVectorizer(
            analyzer="char_wb",
            ngram_range=(3, 5),
            min_df=2,
            sublinear_tf=True
        )
    )
])

pipeline_sgd = Pipeline([
    ("representacion", representacion_sgd),
    (
        "modelo",
        SGDClassifier(
            random_state=42,
            max_iter=5000,
            tol=1e-4
        )
    )
])

pipeline_sgd


## 5. Búsqueda de hiperparámetros

Se prueban tres funciones de pérdida:

- `hinge`: clasificador lineal de margen, similar conceptualmente a una SVM lineal pero optimizado mediante SGD.
- `log_loss`: clasificación logística entrenada mediante SGD.
- `modified_huber`: pérdida robusta que combina propiedades de margen y probabilidades suaves.

`alpha` controla la intensidad de la regularización: valores mayores implican mayor regularización.

La selección se realiza exclusivamente sobre `X_train` mediante validación cruzada estratificada de cinco folds y `accuracy`.


In [ ]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

param_grid_sgd = {
    "modelo__loss": [
        "hinge",
        "log_loss",
        "modified_huber"
    ],
    "modelo__alpha": [
        1e-5,
        3e-5,
        1e-4,
        3e-4
    ],
    "modelo__penalty": ["l2"]
}

grid_sgd = GridSearchCV(
    estimator=pipeline_sgd,
    param_grid=param_grid_sgd,
    scoring="accuracy",
    cv=cv,
    n_jobs=-1,
    verbose=1,
    return_train_score=False
)

grid_sgd.fit(X_train, y_train)

print("Mejores parámetros:", grid_sgd.best_params_)
print(f"Accuracy promedio en CV: {grid_sgd.best_score_:.4f}")


### 5.1. Resultados de la búsqueda

Se muestran todas las combinaciones ordenadas de mayor a menor accuracy promedio.


In [ ]:
resultados = pd.DataFrame(grid_sgd.cv_results_)

tabla_resultados = pd.DataFrame({
    "loss": resultados["param_modelo__loss"],
    "alpha": resultados["param_modelo__alpha"].astype(float),
    "CV media": resultados["mean_test_score"],
    "CV std": resultados["std_test_score"],
    "ranking": resultados["rank_test_score"]
}).sort_values(["ranking", "CV media"])

display(tabla_resultados.reset_index(drop=True).round(5))


## 6. Evaluación final en validación

Después de seleccionar los hiperparámetros mediante CV se evalúa una sola vez sobre `X_val`, que no se utilizó para elegir los parámetros.


In [ ]:
mejor_sgd = grid_sgd.best_estimator_
y_pred = mejor_sgd.predict(X_val)

acc_val = accuracy_score(y_val, y_pred)

print(f"Accuracy en validación: {acc_val:.4f}\n")
print(classification_report(y_val, y_pred, digits=4))

ConfusionMatrixDisplay.from_predictions(
    y_val,
    y_pred,
    cmap="Blues"
)
plt.title("Matriz de confusión: SGDClassifier")
plt.show()


## 7. Comparación de CV y validación

Una diferencia pequeña entre CV y validación sugiere un comportamiento estable. Una diferencia grande debe discutirse antes de considerar un envío a Kaggle.


In [ ]:
resumen_sgd = pd.DataFrame([{
    "Modelo": "SGDClassifier palabras + caracteres",
    "Accuracy CV": grid_sgd.best_score_,
    "Accuracy validación": acc_val,
    "Mejor loss": grid_sgd.best_params_["modelo__loss"],
    "Mejor alpha": grid_sgd.best_params_["modelo__alpha"]
}])

display(resumen_sgd)


## 8. Entrenamiento final con las 12.000 reseñas

Una vez seleccionados los hiperparámetros, el pipeline se vuelve a entrenar utilizando todo `train.csv`. Esto se realiza únicamente para construir el modelo final y las predicciones sobre `eval.csv`.


In [ ]:
from sklearn.base import clone

modelo_sgd_final = clone(mejor_sgd)
modelo_sgd_final.fit(X, y)

carpeta_modelos = raiz / "models"
carpeta_modelos.mkdir(exist_ok=True)

ruta_modelo = carpeta_modelos / "sgd_tfidf_palabras_caracteres.joblib"
joblib.dump(modelo_sgd_final, ruta_modelo)

# Verificar que el modelo guardado puede cargarse
modelo_sgd_cargado = joblib.load(ruta_modelo)

assert np.array_equal(
    modelo_sgd_final.predict(X.iloc[:50]),
    modelo_sgd_cargado.predict(X.iloc[:50])
)

print("Modelo guardado y verificado:", ruta_modelo)


## 9. Generación del CSV para Kaggle

El archivo conserva exactamente los identificadores de `sample_submission.csv` y utiliza únicamente las tres etiquetas permitidas.


In [ ]:
pred_eval = modelo_sgd_cargado.predict(eval_data["text"])

submission_sgd = pd.DataFrame({
    "id": eval_data["id"],
    "answer": pred_eval
})

# Verificaciones de formato
assert list(submission_sgd.columns) == list(sample.columns)
assert submission_sgd["id"].equals(sample["id"])
assert len(submission_sgd) == 3000
assert submission_sgd["answer"].notna().all()
assert set(submission_sgd["answer"]).issubset(set(y.unique()))

carpeta_submissions = raiz / "submissions"
carpeta_submissions.mkdir(exist_ok=True)

ruta_submission = carpeta_submissions / "submission_sgd_tfidf_palabras_caracteres.csv"
submission_sgd.to_csv(ruta_submission, index=False)

print("Archivo generado:", ruta_submission)
print("\nDistribución de predicciones:")
print(submission_sgd["answer"].value_counts())


## 10. Conclusiones

Después de ejecutar el notebook, complete esta sección con los valores reales obtenidos:

- mejor función de pérdida;
- mejor `alpha`;
- accuracy promedio en validación cruzada;
- accuracy en validación;
- principales confusiones observadas en la matriz;
- comparación con los demás modelos del equipo.

El modelo solo debería subirse a Kaggle si el equipo considera útil registrar este experimento o si sus métricas locales son competitivas. Independientemente de que sea el mejor modelo, este notebook documenta un enfoque adicional de clasificación clásica usando TF-IDF de palabras y caracteres con `SGDClassifier`.
